# scRareBench | High-Level Harmony | Dataset 0 — BMMC (GSE194122 curated)

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Google Colab notebook runs **Harmony** on scRareBench **Dataset 0 (GSE194122 paper benchmark)** through the generic **`MethodSpec → benchmark_method`** API.

Harmony remains user-owned method code. scRareBench loads the benchmark dataset, validates latent/cell alignment, evaluates each run, and builds the multi-seed deliverable.

For direct comparability with the current high-level scVI example, the method is evaluated with the same three method seeds:

`42, 123, 2026`

The Harmony-side pipeline follows the existing Dataset 0 Harmony reference workflow:

`raw counts → batch-aware Seurat-v3 HVGs → normalize_total → log1p → scale → PCA → Harmony`

Only the dataset batch key is passed to Harmony. Ground-truth `celltype` labels are not used by the integration method.

> **Runtime note:** Harmony itself is CPU-based. Scaling 89k cells × 4,000 HVGs can require substantial RAM, so a high-memory Colab runtime is recommended.

In [ ]:
# Optional diagnostic only: leave commented unless you want to inspect the runtime.
# import sys
# from importlib import metadata
# print(sys.version)
# for pkg in ("numpy", "pandas", "scipy", "anndata", "scanpy", "harmonypy", "scrarebench"):
#     try:
#         print(pkg, metadata.version(pkg))
#     except metadata.PackageNotFoundError:
#         print(pkg, "not installed")


In [ ]:
import subprocess
import sys

REPO = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@cfdb720f5488b9ceef9a225b35fc7558bcb0251c"

# Install the lightweight package first; setup_runtime handles its runtime dependencies.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", REPO
])

# Method dependencies stay explicitly user-controlled.
METHOD_DEPENDENCIES = ("harmonypy==2.0.0",)
INSTALL_METHOD_DEPENDENCIES = True

from scrarebench.runtime import setup_runtime

runtime_report = setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES if INSTALL_METHOD_DEPENDENCIES else (),
    extra_imports=("harmonypy",) if INSTALL_METHOD_DEPENDENCIES else (),
    quiet=False,
)


## Load Dataset 0

The built-in loader returns the curated Dataset 0 benchmark and its scRareBench metadata. No Harmony-specific preprocessing happens in the dataset loader.


In [ ]:
from scrarebench import load_dataset, dataset_info

adata = load_dataset(0)
info = dataset_info(adata)

assert info.get("dataset_index") == 0
assert info.get("batch_key") in adata.obs.columns

count_layer = info.get("count_layer") or "counts"
assert count_layer in adata.layers, f"Expected raw counts in adata.layers[{count_layer!r}]"

print(adata)
print("Dataset:", info.get("name"))
print("Batch key:", info.get("batch_key"))
print("Label key:", info.get("label_key"))
print("Count layer:", count_layer)
print("Cells:", f"{adata.n_obs:,}")
print("Features:", f"{adata.n_vars:,}")


## Reproducibility and Harmony configuration

`METHOD_SEEDS` matches the current high-level scVI notebook. The benchmark seed remains fixed so that benchmark-side stochastic operations do not change across method seeds.

The parameters below mirror the current detailed Harmony reference notebook for Dataset 0.


In [ ]:
# Same three method seeds used by the high-level scVI example.
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    # Feature selection / preprocessing
    "n_hvg": 4000,
    "normalize_target_sum": 10_000.0,
    "scale_data": True,
    "scale_max_value": 10.0,
    "n_pcs": 50,
    "pca_solver": "arpack",

    # harmonypy 2.0.0 settings
    "theta": 2.0,
    "lamb": None,              # harmonypy 2.0.0: automatic lambda estimation
    "sigma": 0.1,
    "nclust": None,
    "tau": 0.0,
    "block_size": 0.05,
    "max_iter_harmony": 10,
    "max_iter_kmeans": 4,
    "epsilon_cluster": 1e-3,
    "epsilon_harmony": 1e-2,
    "alpha": 0.2,
    "batch_prop_cutoff": 1e-5,
    "ncores": 0,               # 0 = all BLAS threads available to harmonypy
    "verbose": True,
}

print("Method seeds:", METHOD_SEEDS)
print("Benchmark seed:", BENCHMARK_SEED)


## User-owned Harmony method

This is the only Harmony-specific section.

For each seed, the runner:

1. reads Dataset 0 metadata from the AnnData object;
2. keeps RNA/GEX features only when feature-type metadata is available;
3. validates that the declared count layer is suitable for count-based preprocessing;
4. selects 4,000 batch-aware Seurat-v3 HVGs from raw counts;
5. normalizes, log-transforms, scales, and computes PCA;
6. runs Harmony using the dataset batch key only;
7. returns a cell × latent matrix with explicit barcodes.

The ground-truth label column is never passed to Harmony.


In [ ]:
import time

import harmonypy as hm
import numpy as np
import scanpy as sc
from scipy import sparse

from scrarebench import MethodOutput, dataset_info


def _gex_mask(method_adata):
    """Return a robust RNA/GEX feature mask without using benchmark labels."""
    for key in ("feature_types", "feature_type", "modality"):
        if key not in method_adata.var.columns:
            continue

        values = method_adata.var[key].astype(str).str.lower()
        mask = values.str.contains(
            r"gene expression|gex|rna",
            regex=True,
            na=False,
        ).to_numpy()

        if mask.any():
            return mask

    return np.ones(method_adata.n_vars, dtype=bool)


def _validate_count_like(matrix, *, seed, max_values=100_000):
    """Fail early if the declared count layer is clearly not raw/count-like."""
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix).ravel()
    values = np.asarray(values)

    if values.size > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[rng.choice(values.size, size=max_values, replace=False)]

    values = values.astype(float, copy=False)

    if values.size and not np.isfinite(values).all():
        raise ValueError("Raw count layer contains NaN or infinite values.")
    if values.size and np.min(values) < 0:
        raise ValueError("Raw count layer contains negative values.")
    if values.size and not np.allclose(values, np.rint(values), rtol=0.0, atol=1e-6):
        raise ValueError(
            "The declared count layer does not look like integer raw counts."
        )


def run_harmony(method_adata, seed, config):
    meta = dataset_info(method_adata)
    batch_key = str(meta.get("batch_key") or "")
    count_layer = str(meta.get("count_layer") or "counts")

    if not batch_key or batch_key not in method_adata.obs.columns:
        raise KeyError("Dataset batch metadata is missing.")
    if count_layer not in method_adata.layers:
        raise KeyError(
            f"Raw count layer {count_layer!r} is required for Harmony preprocessing."
        )

    seed = int(seed)
    np.random.seed(seed)

    # Keep RNA/GEX features only if the dataset contains multiple modalities.
    feature_mask = _gex_mask(method_adata)
    work = method_adata[:, feature_mask].copy()
    work.var_names_make_unique()

    _validate_count_like(work.layers[count_layer], seed=seed)

    # Batch-aware HVG selection on RAW COUNTS, matching the Dataset 0
    # detailed Harmony reference workflow.
    sc.pp.highly_variable_genes(
        work,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), work.n_vars),
        batch_key=batch_key,
        subset=False,
        check_values=True,
    )

    hvg_mask = work.var["highly_variable"].fillna(False).to_numpy()
    if not hvg_mask.any():
        raise RuntimeError("Harmony preprocessing selected zero HVGs.")

    work = work[:, hvg_mask].copy()

    # Harmony preprocessing starts from the raw-count matrix for the selected HVGs.
    counts = work.layers[count_layer]
    if sparse.issparse(counts):
        work.X = counts.tocsr().astype(np.float32).copy()
    else:
        work.X = np.asarray(counts, dtype=np.float32).copy()

    sc.pp.normalize_total(work, target_sum=float(config["normalize_target_sum"]))
    sc.pp.log1p(work)

    if bool(config["scale_data"]):
        # This usually densifies the matrix. High-memory Colab is recommended.
        sc.pp.scale(work, max_value=float(config["scale_max_value"]))

    n_pcs = min(
        int(config["n_pcs"]),
        work.n_vars - 1,
        work.n_obs - 1,
    )
    if n_pcs < 2:
        raise ValueError(f"Too few dimensions for PCA: {n_pcs}")

    sc.pp.pca(
        work,
        n_comps=n_pcs,
        zero_center=True,
        svd_solver=str(config["pca_solver"]),
        random_state=seed,
    )

    pca = np.asarray(work.obsm["X_pca"], dtype=np.float64)
    if pca.shape != (work.n_obs, n_pcs):
        raise RuntimeError(f"Unexpected PCA shape: {pca.shape}")
    if not np.isfinite(pca).all():
        raise ValueError("PCA representation contains NaN or infinite values.")

    print(
        f"[Harmony seed={seed}] "
        f"cells={work.n_obs:,}, HVGs={work.n_vars:,}, PCs={n_pcs}, "
        f"batches={work.obs[batch_key].nunique()}"
    )

    started = time.perf_counter()
    harmony_out = hm.run_harmony(
        pca,
        work.obs,
        batch_key,
        theta=config["theta"],
        lamb=config["lamb"],
        sigma=config["sigma"],
        nclust=config["nclust"],
        tau=config["tau"],
        block_size=config["block_size"],
        max_iter_harmony=config["max_iter_harmony"],
        max_iter_kmeans=config["max_iter_kmeans"],
        epsilon_cluster=config["epsilon_cluster"],
        epsilon_harmony=config["epsilon_harmony"],
        alpha=config["alpha"],
        batch_prop_cutoff=config["batch_prop_cutoff"],
        verbose=config["verbose"],
        random_state=seed,
        ncores=config["ncores"],
    )
    elapsed = time.perf_counter() - started

    latent = np.asarray(harmony_out.Z_corr, dtype=np.float32)

    if latent.ndim != 2:
        raise RuntimeError(f"Harmony returned a non-2D latent: {latent.shape}")

    # harmonypy 2.0.0 returns cells × PCs. Keep a defensive transpose guard
    # in case a future compatible implementation exposes PCs × cells.
    if latent.shape[0] != work.n_obs and latent.shape[1] == work.n_obs:
        latent = latent.T

    if latent.shape[0] != work.n_obs or latent.shape[1] < 2:
        raise RuntimeError(
            f"Unexpected Harmony latent shape {latent.shape}; "
            f"expected {work.n_obs} rows."
        )
    if not np.isfinite(latent).all():
        raise ValueError("Harmony latent contains NaN or infinite values.")

    print(
        f"[Harmony seed={seed}] latent={latent.shape}, "
        f"integration_time={elapsed:.2f}s"
    )

    return MethodOutput(
        latent=latent,
        barcodes=work.obs_names,
    )


## Run the three-seed high-level benchmark

`benchmark_method` runs the complete Harmony runner independently for each method seed and keeps the benchmark-side random seed fixed at 42.

No method latent cache is used in this notebook: all three seeds are genuinely executed.


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="Harmony",
    runner=run_harmony,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    benchmark_config={"random_state": BENCHMARK_SEED},
    # Dependencies were already handled above by setup_runtime.
    install_dependencies=False,
    finalize=True,
)


## Inspect the multi-seed result


In [ ]:
from IPython.display import display

display(result.summary().round(5))

print("Completed method seeds:", result.seeds)
print("Multi-seed report:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)


## Optional: inspect each seed separately


In [ ]:
for seed in result.seeds:
    run = result.runs[seed]
    print(f"\n===== Harmony seed {seed} =====")
    print("Output directory:", run.output_dir)
    try:
        display(run.summary().round(5))
    except Exception:
        # Keep this diagnostic cell compatible if a single-run result object
        # exposes its tables through attributes rather than summary().
        print(run)


## Download the complete output archive


In [ ]:
from pathlib import Path
from google.colab import files

archive_path = Path(result.archive_path)

if not archive_path.exists():
    raise FileNotFoundError(
        f"Output archive was not found: {archive_path}\n"
        "Make sure benchmark_method(..., finalize=True) completed successfully."
    )

print(f"Downloading: {archive_path.name}")
print(f"Size: {archive_path.stat().st_size / (1024**2):.2f} MB")

files.download(str(archive_path))
